# LARA Training — Google Colab
Modifie `EXPERIMENT` ci-dessous puis exécute toutes les cellules.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────
EXPERIMENT   = "diff_attn"  # diff_attn | mor | coconut | lara_full | lara_v2_full | lara_v2_dca
MAX_ITERS    = 5000
BATCH_SIZE   = 8
GRAD_ACCUM   = 16
N_EMBD       = 1024
N_LAYER      = 6
N_HEAD       = 8
BLOCK_SIZE   = 512
LR           = "3e-4"
WARMUP       = 500
N_RECURSIONS = 4
WANDB        = False
# ──────────────────────────────────────────────────────────────

RUN_NAMES = {
    'baseline':     'exp_a_baseline',
    'diff_attn':    'exp_b_diff_attn',
    'mor':          'exp_c_mor',
    'coconut':      'exp_d_coconut',
    'lara_full':    'exp_e_lara_full',
    'lara_v2':      'exp_f_lara_v2',
    'lara_v2_dca':  'exp_h_lara_v2_dca',
    'lara_v2_full': 'exp_g_lara_v2_full',
}
RUN_NAME = RUN_NAMES[EXPERIMENT]
print(f'Expérience : {EXPERIMENT} → checkpoint : {RUN_NAME}_best.pt')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/LARA_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints Drive : {DRIVE_DIR}')

In [ ]:
import os
if os.path.exists('/content/LARA'):
    !git -C /content/LARA pull
else:
    !git clone https://github.com/s3basti3nDev/LARA.git /content/LARA

if os.path.exists('/content/LARA/lara/train.py'):
    LARA_DIR = '/content/LARA/lara'
elif os.path.exists('/content/LARA/train.py'):
    LARA_DIR = '/content/LARA'
else:
    raise FileNotFoundError('train.py introuvable')

os.chdir(LARA_DIR)
print(f'Dossier : {LARA_DIR}')
!ls

In [ ]:
!pip install tiktoken datasets wandb -q
!nvidia-smi | grep -E 'GPU|Memory'

In [ ]:
# Entraînement avec affichage en temps réel
import subprocess, sys, os

cmd = [
    sys.executable, '-u', 'train.py',
    '--experiment',    EXPERIMENT,
    '--dataset',       'fineweb',
    '--n_embd',        str(N_EMBD),
    '--n_layer',       str(N_LAYER),
    '--n_head',        str(N_HEAD),
    '--block_size',    str(BLOCK_SIZE),
    '--learning_rate', LR,
    '--warmup_iters',  str(WARMUP),
    '--batch_size',    str(BATCH_SIZE),
    '--grad_accum',    str(GRAD_ACCUM),
    '--max_iters',     str(MAX_ITERS),
    '--device',        'cuda',
    '--compile',
]
if EXPERIMENT in ('lara_v2', 'lara_v2_dca', 'lara_v2_full'):
    cmd += ['--n_recursions', str(N_RECURSIONS)]
if WANDB:
    cmd.append('--wandb')

print('Lancement :', ' '.join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Entraînement échoué (code {proc.returncode})')

In [ ]:
# Copier le checkpoint vers Google Drive
import shutil
src = f'checkpoints/{RUN_NAME}_best.pt'
dst = f'{DRIVE_DIR}/{RUN_NAME}_best.pt'
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'Checkpoint sauvegardé : {dst}')
else:
    print('Checkpoints disponibles :', os.listdir('checkpoints') if os.path.exists('checkpoints') else 'aucun')

In [ ]:
# Evaluer
proc2 = subprocess.Popen([sys.executable, '-u', 'evaluate.py', '--model', dst, '--dataset', 'fineweb'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc2.stdout:
    print(line, end='', flush=True)
proc2.wait()